<style>
div.mermaid > svg { width: 70% !important; height: auto !important; }
</style>

# Array concepts: `cutlass.Array` and the GPU memory spaces

`cutlass.Array` is the workhorse memory type of this course: a typed handle to GPU memory that
knows its **shape**, **strides**, **dtype**, and **memory space**. One handle serves scalar `a[i]`,
multi-dimensional `a[r, c]`, and offset `subview` access — no hand-written pointer arithmetic.

The other half is **where** the memory lives. One `Array` type covers four GPU memory spaces, and
the space decides **who owns it, how long it lives, and which threads see it**. Two kernels carry
the chapter: one allocates arrays **inside** the kernel, the other uses arrays declared **outside**
it at module scope. Each prints the arrays it touches.

**You'll learn:** the central data type of the DSL, **`cutlass.Array`**. Kernel arguments are
Arrays; the **slice = vectorized load/store** idiom (`a[idx:V]` moves `V` contiguous elements in
one wide transaction once you promise alignment); how to read an Array's printed layout (dtype,
space, shape, alignment); the **local / shared / global / constant** memory spaces and their
ownership, lifetime, and visibility; allocating **local** and **shared** scratch inside a kernel
(with the barrier that makes shared safe); declaring **global** and **constant** statics at module
scope; and `subview`. We build a complete vectorized vector-add along the way.

**Runs on:** any CUDA GPU. (Registers as *values* are `cutlass.Vector`, in `02_vector_concepts`;
the `cute.Tensor` -> `cutlass.Array` bridge is `03_cute_interop`.)

In [ ]:
import cutlass
import cutlass.cute as cute
import torch

## The four spaces

`cutlass.Array(dtype, size, space=...)` picks its allocator from the **space**. Each maps to a
familiar CUDA C++ declaration:

| `cutlass.Array(...)` | CUDA C++ analog | space | who sees it / lifetime |
|---|---|---|---|
| `cutlass.Array(f32, N)` | `float a[N];` | **local** (default) | one private copy per **thread** |
| `cutlass.Array(f32, N, space=…smem)` | `__shared__ float a[N];` | **shared** | one copy per **block** |
| `cutlass.Array(f32, N, space=…gmem, name="g")` | `__device__ float g[N];` | **global** | one **static** buffer; all threads + launches share it |
| `cutlass.Array(f32, N, space=…cmem, name="c", init=…)` | `__constant__ float c[N];` | **constant** | read-only static, broadcast to all threads |

**local** and **shared** are scratch you allocate **inside** a kernel; **global** and **constant**
are **statics** you declare at module scope and use by name. `size` is always a compile-time
constant — these are static buffers, not a per-thread `malloc`. Global memory also arrives the
everyday way, as a **kernel argument** (a tensor via `from_dlpack`).

## 1. One thread, a vector of elements

The kernel is three lines, so the only design decision is *how much work each thread does*. We give
every thread a window of `V` **contiguous** elements and let it process the whole window at once.

The mechanism is the slice. `a[idx:V]` reads `V` elements starting at `idx`; assigning to `c[idx:V]`
writes them back. The second number is a **count**, not a Python stop index -- `a[idx:V]` means "`V`
elements from `idx`", not "elements `idx` up to `V`". So a thread owning the span `[idx, idx+V)`
loads both inputs, adds them register-to-register, and stores the result.

Whether that slice becomes **one wide transaction** or `V` scalar ones comes down to **alignment**. A
128-bit load of four `f32`s is only legal from a 16-byte-aligned address, and the compiler will not
assume that on its own. We make the promise when we wrap the buffer below
(`from_dlpack(a, assumed_align=16)`), and it genuinely holds: every thread's window starts at
`idx = (block * block_size + thread) * V`, a multiple of `V = 4` elements, i.e. 16 bytes.

| | one 128-bit transaction | `V` scalar transactions |
|---|---|---|
| when | `assumed_align=16` promised | no alignment promise |
| `a[idx:V]` lowers to | `ld.global.v4.b32` / `v2.b64` | four `ld.global.b32` |
| per slice | **1** instruction | **4** instructions |

The exact wide opcode is GPU-specific -- `ld.global.v4.b32` (four 32-bit words) on Ampere/Hopper, or
`ld.global.v2.b64` (two 64-bit words) on Blackwell, which packs the four values so one `add.f32x2`
adds two at a time. Same 128 bits either way; section 4 prints whichever your GPU uses.

In [ ]:
@cute.kernel
def vector_add_kernel(
    a: cutlass.Array, b: cutlass.Array, c: cutlass.Array, N: cutlass.Int32, V: cutlass.Constexpr
):
    # Step 1. Map this thread to the first element of its V-wide window. The global
    # lane is (block * block_size + thread); scaling by V gives the window's start.
    tx, _, _ = cute.arch.thread_idx()
    bx, _, _ = cute.arch.block_idx()
    bdx, _, _ = cute.arch.block_dim()
    idx = (bx * bdx + tx) * V

    # Step 2. Guard the tail, then add one V-wide window. The grid rounds up to whole
    # blocks, so the last block has threads whose window starts past the array; skip them.
    if idx < N:
        c[idx:V] = a[idx:V] + b[idx:V]

## 2. Launch: one thread per window

Each thread handles `V` elements, so we need `N / V` threads. We pick a 1-D block of 256 threads (a
conventional, occupancy-friendly size) and round the grid up so no element is left uncovered -- the
in-kernel bounds check absorbs the few extra threads. We take `N` to be a multiple of `V`, so every
active thread's window lands fully inside the array.

In [ ]:
@cute.jit
def vector_add(
    a: cutlass.Array, b: cutlass.Array, c: cutlass.Array, N: cutlass.Int32, V: cutlass.Constexpr
):
    block = (256, 1, 1)
    threads = N // V  # one thread per V-wide window
    grid = ((threads + block[0] - 1) // block[0], 1, 1)  # round up to whole blocks

    vector_add_kernel(a, b, c, N, V).launch(grid=grid, block=block)

## 3. Run it and check against PyTorch

`cutlass.Array` parameters are host-entry types, so there is no manual buffer setup: hand the PyTorch
CUDA tensors straight to the kernel through `cute.runtime.from_dlpack`, which wraps the existing GPU
memory with zero copies. We then check the result against PyTorch's own `a + b`.

In [ ]:
N, V = 1 << 20, 4  # 1,048,576 elements, 4 per thread

# The grid is sized as N // V, so V must divide N exactly. Otherwise the trailing
# elements would have no thread assigned, and a thread's window could run past the end.
assert N % V == 0, "N must be a multiple of the vector width V"

a = torch.randn(N, dtype=torch.float32, device="cuda")
b = torch.randn(N, dtype=torch.float32, device="cuda")
c = torch.zeros(N, dtype=torch.float32, device="cuda")

# from_dlpack wraps each CUDA tensor as a cutlass.Array with no copy. assumed_align=16
# promises 16-byte-aligned buffers (torch's allocator guarantees far more), which is what
# lets the V=4 slice load/store become single 128-bit transactions (see section 4).
a_ = cute.runtime.from_dlpack(a, assumed_align=16)
b_ = cute.runtime.from_dlpack(b, assumed_align=16)
c_ = cute.runtime.from_dlpack(c, assumed_align=16)

vector_add(a_, b_, c_, N, V)

torch.testing.assert_close(c.cpu(), (a + b).cpu(), atol=1e-5, rtol=1e-5)
print("PASS")

# Expected output:
# PASS

## 4. Prove it vectorized -- straight from the PTX

We claimed the `V = 4` slice becomes one 128-bit transaction. `cute.compile[cute.KeepPTX]` keeps the
generated PTX on the result's `.artifacts.PTX`, so we read it back and look at just the global loads
and stores. Compiling the *same* kernel **with** and **without** the alignment promise shows the
difference directly: three vector instructions versus twelve scalar ones. (The wide opcode is
GPU-specific -- `v4.b32` on Ampere/Hopper, `v2.b64` on Blackwell -- but both are one 128-bit
transaction, so the *count* is the point.)

In [ ]:
def global_mem_ops(compiled):
    """The global load/store instructions in a compiled kernel's PTX."""
    return [
        ln.strip()
        for ln in compiled.artifacts.PTX.splitlines()
        if "ld.global" in ln or "st.global" in ln
    ]


print("WITH the 16-byte promise (assumed_align=16):")
for op in global_mem_ops(cute.compile[cute.KeepPTX](vector_add, a_, b_, c_, N, V)):
    print("   ", op)

# Same kernel, same data -- but no alignment promise, so the compiler stays conservative.
plain = [cute.runtime.from_dlpack(t) for t in (a, b, c)]
print("\nWITHOUT the promise:")
for op in global_mem_ops(cute.compile[cute.KeepPTX](vector_add, *plain, N, V)):
    print("   ", op)

# Expected output -- the exact opcode is GPU-specific (v4.b32 on Ampere/Hopper, v2.b64 on
# Blackwell); both are one 128-bit transaction, so the *count* is the point:
# WITH the 16-byte promise (assumed_align=16):   -> 3 vector ops
#     ld.global.{v4.b32 | v2.b64} ...   <- one 128-bit load per operand (a and b)
#     ld.global.{v4.b32 | v2.b64} ...
#     st.global.{v4.b32 | v2.b64} ...   <- one 128-bit store
# WITHOUT the promise:                            -> 12 scalar ops
#     ld.global.b32 ...   <- eight 32-bit loads (four per operand)
#     st.global.b32 ...   <- four 32-bit stores

## 1. Arrays inside `@cute.kernel`

Inside a kernel you allocate scratch: a per-block **shared** tile and a per-thread **local** buffer.
This circular 3-tap row blur uses both. It reads a row from a **global** argument, stages it in
shared so each thread can reach its neighbors' columns, gathers three taps into local, and writes
the average back. It prints each array's layout first — a one-time, compile-time print.

In [ ]:
@cute.kernel
def blur_kernel(inp: cutlass.Array, out: cutlass.Array, C: cutlass.Constexpr):
    row, _, _ = cute.arch.block_idx()   # one block per row
    col, _, _ = cute.arch.thread_idx()  # one thread per column

    # Step 1. Allocate scratch and print each array's layout (a one-time, compile-time print).
    tile = cutlass.Array(inp.dtype, C, space=cutlass.AddressSpace.smem)  # per-block scratch
    taps = cutlass.Array(inp.dtype, 3)                                   # per-thread scratch
    print("arg    :", inp)        # a global Array (the kernel argument)
    print("shared :", tile)
    print("local  :", taps)

    # Step 2. Stage this row from global into shared; the barrier makes it block-visible.
    row_in = inp.subview(row * inp.strides[0])   # subview: an offset view of this row
    tile[col] = row_in[col]                      # global -> shared
    cute.arch.barrier()                     # row now visible to every thread

    # Step 3. Gather three taps into local and write the average back.
    taps[0] = tile[(col + C - 1) % C]            # left neighbor (another thread wrote it)
    taps[1] = tile[col]
    taps[2] = tile[(col + 1) % C]                # right neighbor
    out[row, col] = (taps[0] + taps[1] + taps[2]) / 3.0

In [ ]:
@cute.jit
def blur(inp: cutlass.Array, out: cutlass.Array, N: cutlass.Int32, C: cutlass.Constexpr):
    blur_kernel(inp, out, C).launch(grid=(N, 1, 1), block=(C, 1, 1))


N, C = 1024, 256
inp = torch.randn(N, C, dtype=torch.float32, device="cuda")
out = torch.zeros_like(inp)

blur(cute.runtime.from_dlpack(inp), cute.runtime.from_dlpack(out), N, C=C)

ref = (inp.roll(1, dims=1) + inp + inp.roll(-1, dims=1)) / 3.0   # circular 3-tap blur
torch.testing.assert_close(out.cpu(), ref.cpu(), atol=1e-5, rtol=1e-5)
print("PASS")

# Expected output (the layout prints fire once, while the kernel is staged):
# arg    : Array(dtype=Float32, address_space=gmem, shape=(1024, 256), alignment=4)
# shared : Array(dtype=Float32, address_space=smem, shape=(256,), alignment=4)
# local  : Array(dtype=Float32, address_space=generic, shape=(3,), alignment=4)
# PASS
#
# Only the local array reports `generic`, and that is correct -- not a mislabel: per-thread
# scratch is an `alloca`, stack memory addressed through a *generic* pointer, so one load/store
# path serves it. The dedicated spaces (gmem/smem/cmem) each carry their own address space and
# report it by name.

## 2. Arrays outside `@cute.kernel`

**global** and **constant** arrays are program-wide statics, so you declare them once at **module
scope** — outside any kernel — and use them by name, like CUDA C++ `__device__` / `__constant__`.
(Module-scope statics work as soon as you `import cutlass` — no extra setup.)

`GAMMA` is a read-only constant lookup table (baked in with `init=`); `RUN_COUNT` is a global
counter that **persists across launches**. The kernel scales its input by `GAMMA` and bumps
`RUN_COUNT`. A static is a `_GlobalVariable` handle that becomes an `Array` when you index it, so we
print `.subview(0)` (the whole buffer) to see it as an `Array`.

In [ ]:
# Declared OUTSIDE any kernel, at module scope:
GAMMA = cutlass.Array(cutlass.Float32, 4, name="gamma",
                      init=[1.0, 1.5, 2.0, 2.5], space=cutlass.AddressSpace.cmem)
RUN_COUNT = cutlass.Array(cutlass.Int32, 1, name="run_count", space=cutlass.AddressSpace.gmem)


@cute.kernel
def scale_kernel(inp: cutlass.Array, out: cutlass.Array, seen: cutlass.Array):
    i, _, _ = cute.arch.thread_idx()

    # Step 1. Materialize each module-scope static as an Array and print its layout.
    print("constant:", GAMMA.subview(0))     # materialize the static -> Array
    print("global  :", RUN_COUNT.subview(0))

    # Step 2. Scale the input by the module-scope constant table.
    out[i] = inp[i] * GAMMA[i]                          # read the module-scope constant

    # Step 3. Thread 0 bumps the module-scope global counter (it persists across launches).
    if i == 0:
        RUN_COUNT[0] = RUN_COUNT[0] + cutlass.Int32(1)  # bump the module-scope global counter
        seen[0] = RUN_COUNT[0]


@cute.jit
def scale(inp: cutlass.Array, out: cutlass.Array, seen: cutlass.Array):
    scale_kernel(inp, out, seen).launch(grid=(1, 1, 1), block=(4, 1, 1))

In [ ]:
g_in = torch.tensor([2.0, 4.0, 6.0, 8.0], device="cuda")
g_out = torch.zeros(4, device="cuda")
seen = torch.zeros(1, dtype=torch.int32, device="cuda")

scale(cute.runtime.from_dlpack(g_in), cute.runtime.from_dlpack(g_out), cute.runtime.from_dlpack(seen))

torch.testing.assert_close(g_out.cpu(), torch.tensor([2.0, 6.0, 12.0, 20.0]))  # inp * GAMMA
assert seen.item() == 1                                                         # counter went 0 -> 1
print("PASS")

# Expected output (layout prints fire once, while the kernel is staged):
# constant: Array(dtype=Float32, address_space=cmem, shape=(4,), alignment=4)
# global  : Array(dtype=Int32, address_space=gmem, shape=(1,), alignment=4)
# PASS

## 5. Bonus — the same kernel as a decorator

The kernel-plus-launch above is boilerplate: identical for *any* element-wise op. A small
`@elementwise` **decorator** captures it once, so a one-line op becomes a full vectorized kernel.
The op is ordinary Python that runs at trace time -- `x + y` here builds the very same `V`-wide add,
slice load/store and all.

In [ ]:
def elementwise(op):
    """Lift a binary op into a vectorized kernel + launcher (one V-wide window per thread)."""

    @cute.kernel
    def kern(a: cutlass.Array, b: cutlass.Array, c: cutlass.Array,
             N: cutlass.Int32, V: cutlass.Constexpr):
        tx, _, _ = cute.arch.thread_idx()
        bx, _, _ = cute.arch.block_idx()
        bdx, _, _ = cute.arch.block_dim()
        idx = (bx * bdx + tx) * V
        if idx < N:
            c[idx:V] = op(a[idx:V], b[idx:V])   # op runs at trace time, on the V-wide vectors

    @cute.jit
    def host(a: cutlass.Array, b: cutlass.Array, c: cutlass.Array,
             N: cutlass.Int32, V: cutlass.Constexpr):
        threads = N // V
        grid = ((threads + 255) // 256, 1, 1)
        kern(a, b, c, N, V).launch(grid=grid, block=(256, 1, 1))

    return host


@elementwise
def vadd(x, y):
    return x + y   # the whole op -- the decorator supplies the kernel and the launch


vadd(a_, b_, c_, N, V)
torch.testing.assert_close(c.cpu(), (a + b).cpu(), atol=1e-5, rtol=1e-5)
print("PASS  (decorator-generated kernel matches)")

# Expected output:
# PASS  (decorator-generated kernel matches)

## Try it yourself

1. **Drop the barrier.** Remove `cute.arch.barrier()` from `blur_kernel` — the blur goes wrong
   and nondeterministic, because a thread may read a neighbor's column before that neighbor wrote it.
2. **Make the tile local.** Drop `space=…smem` from `tile` (and the barrier): each thread gets its
   *own* `tile` with only `tile[col]` written, so neighbor reads hit uninitialized memory — the
   difference between shared (block-wide) and local (per-thread).
3. **Inside or outside.** Move `GAMMA`'s declaration *inside* `scale_kernel` — it still works: a
   named static is the same buffer wherever you declare it. Then launch `scale` twice and watch
   `RUN_COUNT` climb to 2 — a global static persists across launches.
4. **Constant is read-only.** Store into `GAMMA` from a kernel — it raises `TypeError`, because
   constant memory cannot be written from the device. (And `tmem` / `dsmem` from the `AddressSpace`
   enum are not usable through `cutlass.Array`; they have dedicated Blackwell-chapter APIs.)